# Phase 2 - UsnJrnl Feature Engineering (LoneWolf Test)

## Objective
Test feature engineering pipeline on LoneWolf dataset to validate scoring logic before scaling to all datasets.

## Approach
1. Load enriched UsnJrnl from Phase 0
2. Filter for relevant events
3. Group by file and calculate file-level features
4. Calculate event-level features
5. Label events from ground truth
6. Validate behavior counter logic

---

In [9]:
## Cell 1: Setup and Load Data
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
USNJRNL_PATH = BASE_DIR / 'data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched.csv'
SUSPICIOUS_PATH = BASE_DIR / 'data/validation/suspicious/LoneWolf-Suspicious.csv'
OUTPUT_PATH = BASE_DIR / 'data/validation/processed/phase 2/LoneWolf_UsnJrnl_features.csv'

# Create output directory
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Load data
usnjrnl = pd.read_csv(USNJRNL_PATH)
suspicious = pd.read_csv(SUSPICIOUS_PATH)

print(f"UsnJrnl Records: {len(usnjrnl):,}")
print(f"Suspicious Events: {len(suspicious):,}")
print(f"\nColumns: {list(usnjrnl.columns)}")


UsnJrnl Records: 352,849
Suspicious Events: 15

Columns: ['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'FileReferenceNumber', 'FileName_MFT', 'SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 'SI_AccessedTime_Formatted', 'SI_MFTModifiedTime_Formatted', 'EventTime_Formatted', 'MFT_RecordNumber']


In [13]:
# Cell 2: Aggressive Event Filtering (Optimized for Scale)

print("Applying aggressive filtering - keeping only detection-critical events...")

# Step 1: Keep ALL Basic_Info_Changed events (potential timestomping)
bdp_events = usnjrnl[
    usnjrnl['EventInfo'].str.contains('Basic_Info_Changed', na=False)
].copy()

# Step 2: Keep FIRST File_Created event per file (for creation time)
creation_events = usnjrnl[
    usnjrnl['EventInfo'].str.contains('File_Created', na=False)
].sort_values('TimeStamp(UTC+8)').groupby('FileReferenceNumber').first().reset_index()

# Step 3: Keep Delete/Rename events ONLY at paths where we have creation events
# (Tunneling can only affect files we're monitoring)
monitored_paths = set(creation_events['FullPath'].dropna().unique())
print(f"  Monitoring {len(monitored_paths):,} unique paths")

delete_rename_events = usnjrnl[
    (usnjrnl['EventInfo'].str.contains('File_Deleted|File_Renamed', na=False)) &
    (usnjrnl['FullPath'].isin(monitored_paths))
].copy()


# Step 4: Combine and deduplicate
usnjrnl_filtered = pd.concat([
    bdp_events,
    creation_events,
    delete_rename_events
]).drop_duplicates(subset=['USN']).copy()

print(f"\nOriginal events: {len(usnjrnl):,}")
print(f"Filtered events: {len(usnjrnl_filtered):,}")
print(f"Reduction: {(1 - len(usnjrnl_filtered)/len(usnjrnl))*100:.1f}%")

print(f"\nBreakdown:")
print(f"  Basic_Info_Changed events: {len(bdp_events):,}")
print(f"  First File_Created per file: {len(creation_events):,}")
print(f"  Delete/Rename at monitored paths: {len(delete_rename_events):,}")

print(f"\nEstimated combined dataset size (22 datasets):")
estimated_total = len(usnjrnl_filtered) * 22
print(f"  {estimated_total:,} events (~{estimated_total/1000000:.1f}M)")

print(f"\nEvent type distribution after filtering:")
print(usnjrnl_filtered['EventInfo'].value_counts().head(10))


Applying aggressive filtering - keeping only detection-critical events...
  Monitoring 18,941 unique paths

Original events: 352,849
Filtered events: 133,959
Reduction: 62.0%

Breakdown:
  Basic_Info_Changed events: 33,713
  First File_Created per file: 54,939
  Delete/Rename at monitored paths: 50,102

Estimated combined dataset size (22 datasets):
  2,947,098 events (~2.9M)

Event type distribution after filtering:
EventInfo
File_Closed / File_Deleted                                      29807
File_Created                                                    26729
File_Created / Data_Added / File_Closed                         11421
File_Created / File_Closed                                       9848
Basic_Info_Changed                                               9224
Basic_Info_Changed / File_Closed                                 5217
File_Renamed_New                                                 4742
File_Renamed_New / File_Closed                                   4736
Basic_Inf

In [11]:
import pandas as pd

# Load the CSV
path = "/Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 2/LoneWolf_UsnJrnl_features.csv"
df = pd.read_csv(path)

# Show all rows when printing
pd.set_option('display.max_rows', None)

# Count instances of each EventInfo value
event_counts = df["EventInfo"].value_counts()
print(event_counts)


EventInfo
File_Created                                                                                                                      54905
File_Closed / File_Deleted                                                                                                        48184
File_Created / Data_Added                                                                                                         29402
File_Created / Data_Added / File_Closed                                                                                           24698
File_Created / File_Closed                                                                                                        22690
Basic_Info_Changed                                                                                                                 9224
File_Renamed_Old                                                                                                                   8982
File_Renamed_New                      

In [3]:
# Cell 3: Parse Timestamps
# Parse timestamps to datetime
usnjrnl_filtered['EventTime'] = pd.to_datetime(
    usnjrnl_filtered['TimeStamp(UTC+8)'],
    format='%m/%d/%y %H:%M:%S:%f',
    errors='coerce'
)

usnjrnl_filtered['SI_CreationTime'] = pd.to_datetime(
    usnjrnl_filtered['SI_CreationTime_Formatted'],
    format='%m/%d/%Y %H:%M:%S.%f',
    errors='coerce'
)

usnjrnl_filtered['SI_ModifiedTime'] = pd.to_datetime(
    usnjrnl_filtered['SI_ModifiedTime_Formatted'],
    format='%m/%d/%Y %H:%M:%S.%f',
    errors='coerce'
)

usnjrnl_filtered['SI_AccessedTime'] = pd.to_datetime(
    usnjrnl_filtered['SI_AccessedTime_Formatted'],
    format='%m/%d/%Y %H:%M:%S.%f',
    errors='coerce'
)

print("Timestamps parsed successfully")
print(f"\nSample timestamps:")
print(usnjrnl_filtered[['EventTime', 'SI_CreationTime', 'SI_ModifiedTime']].head())


Timestamps parsed successfully

Sample timestamps:
                EventTime         SI_CreationTime         SI_ModifiedTime
3 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772 2018-04-01 16:56:08.779
4 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772 2018-04-01 16:56:08.779
5 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772 2018-04-01 16:56:08.779
6 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772 2018-04-01 16:56:08.779
7 2018-04-01 16:56:08.568 2018-04-01 16:56:08.792 2018-04-01 16:56:08.800


In [4]:
# Cell 4: Calculate File-Level Features 
# Group by FileReferenceNumber and calculate file-level features
file_features = {}

for frn, group in usnjrnl_filtered.groupby('FileReferenceNumber'):
    
    # Sort by timestamp
    group = group.sort_values('EventTime')
    
    # File-level features
    features = {
        'file_event_count': len(group),
        'file_time_span_seconds': (group['EventTime'].max() - group['EventTime'].min()).total_seconds(),
    }
    
    # Check for File_Created events
    creation_events = group[group['EventInfo'].str.contains('File_Created', na=False)]
    if len(creation_events) > 0:
        features['file_has_creation'] = 1
        features['file_creation_time'] = creation_events.iloc[0]['EventTime']
    else:
        features['file_has_creation'] = 0
        features['file_creation_time'] = None
    
    # Find last BDP event
    bdp_events = group[
        (group['EventInfo'].str.contains('Basic_Info_Changed', na=False)) &
        (group['EventInfo'].str.contains('File_Closed', na=False))
    ]
    if len(bdp_events) > 0:
        features['file_last_bdp_timestamp'] = bdp_events.iloc[-1]['EventTime']
        features['file_has_bdp'] = 1
    else:
        features['file_last_bdp_timestamp'] = None
        features['file_has_bdp'] = 0
    
    # Initialize behavior counter (will calculate in next step)
    features['file_behavior_counter'] = 0
    features['file_has_tunneling'] = 0
    
    file_features[frn] = features

# Convert to DataFrame
file_features_df = pd.DataFrame.from_dict(file_features, orient='index')
file_features_df.index.name = 'FileReferenceNumber'

print(f"File-level features calculated for {len(file_features_df):,} unique files")
print(f"\nSample file features:")
print(file_features_df.head())


File-level features calculated for 66,373 unique files

Sample file features:
                     file_event_count  file_time_span_seconds  \
FileReferenceNumber                                             
0x0001000000000546                  1                     0.0   
0x0001000000000547                  1                     0.0   
0x0001000000000548                  1                     0.0   
0x0001000000000549                  1                     0.0   
0x000100000000054A                  1                     0.0   

                     file_has_creation file_creation_time  \
FileReferenceNumber                                         
0x0001000000000546                   0                NaT   
0x0001000000000547                   0                NaT   
0x0001000000000548                   0                NaT   
0x0001000000000549                   0                NaT   
0x000100000000054A                   0                NaT   

                    file_last_bdp_time

In [5]:
# Cell 5: Detect File System Tunneling (EFFICIENT - Vectorized)
import time

print("Detecting file system tunneling patterns (vectorized approach)...")
start_time = time.time()

# Filter to only creation events (we only check tunneling for newly created files)
creation_events = usnjrnl_filtered[
    usnjrnl_filtered['EventInfo'].str.contains('File_Created', na=False)
].copy()

# Filter to only delete/rename events (potential tunneling sources)
delete_rename_events = usnjrnl_filtered[
    usnjrnl_filtered['EventInfo'].str.contains('File_Deleted|File_Renamed', na=False)
].copy()

print(f"  Creation events: {len(creation_events):,}")
print(f"  Delete/Rename events: {len(delete_rename_events):,}")

# Merge on FullPath to find same-path events
tunneling_candidates = creation_events.merge(
    delete_rename_events,
    on='FullPath',
    how='inner',
    suffixes=('_created', '_deleted')
)

print(f"  Same-path pairs: {len(tunneling_candidates):,}")

# Filter to different FileReferenceNumbers
tunneling_candidates = tunneling_candidates[
    tunneling_candidates['FileReferenceNumber_created'] != tunneling_candidates['FileReferenceNumber_deleted']
]

print(f"  Different file pairs: {len(tunneling_candidates):,}")

# Calculate time difference (creation - delete/rename)
tunneling_candidates['time_diff'] = (
    tunneling_candidates['EventTime_created'] - tunneling_candidates['EventTime_deleted']
).dt.total_seconds()

# Tunneling = delete/rename happened 0-15 seconds BEFORE creation
tunneling_pairs = tunneling_candidates[
    (tunneling_candidates['time_diff'] > 0) &
    (tunneling_candidates['time_diff'] <= 15)
]

print(f"  Tunneling pairs (within 15s): {len(tunneling_pairs):,}")

# Get unique FileReferenceNumbers that have tunneling
tunneling_frns = set(tunneling_pairs['FileReferenceNumber_created'].unique())

# Update file_features_df
file_features_df['file_has_tunneling'] = 0
file_features_df['file_has_tunneling'] = file_features_df.index.isin(tunneling_frns).astype(int)

elapsed = time.time() - start_time
tunneling_count = int(file_features_df['file_has_tunneling'].sum())

print(f"\nTunneling detection complete in {elapsed:.1f} seconds")
print(f"Files with tunneling: {tunneling_count:,} ({tunneling_count/len(file_features_df)*100:.1f}%)")
print(f"Files without tunneling: {len(file_features_df) - tunneling_count:,}")


Detecting file system tunneling patterns (vectorized approach)...
  Creation events: 146,838
  Delete/Rename events: 114,539


KeyboardInterrupt: 

In [ ]:
# Cell 6: Calculate Behavior Counter
# Calculate behavior counter for each file
for frn, group in usnjrnl_filtered.groupby('FileReferenceNumber'):
    
    counter = 0
    
    # Get file features
    file_info = file_features_df.loc[frn]
    
    # Get SI_CreationTime (use first non-null value)
    si_creation_time = group['SI_CreationTime'].dropna().iloc[0] if len(group['SI_CreationTime'].dropna()) > 0 else None
    
    if si_creation_time is None:
        continue
    
    # Case 1: If file has creation event
    if file_info['file_has_creation'] == 1 and file_info['file_creation_time'] is not None:
        
        # Calculate difference between SI-C and creation event time
        creation_diff = abs((si_creation_time - file_info['file_creation_time']).total_seconds())
        
        # If difference > 5 seconds, increment counter
        if creation_diff > 5:
            counter += 1
    
    # Case 2: Check last BDP timestamp
    if file_info['file_last_bdp_timestamp'] is not None:
        
        # Calculate difference between SI-C and last BDP
        bdp_diff = abs((si_creation_time - file_info['file_last_bdp_timestamp']).total_seconds())
        
        # If difference > 5 seconds, increment counter
        if bdp_diff > 5:
            counter += 1
    
    # Case 3: Tunneling adjustment
    if counter >= 1:  # Only check tunneling if already suspicious
        if file_info['file_has_tunneling'] == 1:
            counter -= 1  # Reduce suspicion
        else:
            counter += 2  # Increase suspicion
    
    # Update counter
    file_features_df.loc[frn, 'file_behavior_counter'] = counter

print("Behavior counter calculation complete")
print(f"\nBehavior counter distribution:")
print(file_features_df['file_behavior_counter'].value_counts().sort_index())


Behavior counter calculation complete

Behavior counter distribution:
file_behavior_counter
0    26204
1     4016
3    35454
4      699
Name: count, dtype: int64


In [ ]:
# Cell 7: Join Features Back to Events
# Join file-level features back to each event
usnjrnl_with_features = usnjrnl_filtered.merge(
    file_features_df,
    on='FileReferenceNumber',
    how='left'
)

print(f"Events with file features: {len(usnjrnl_with_features):,}")
print(f"\nNew columns added:")
print([col for col in usnjrnl_with_features.columns if col.startswith('file_')])


Events with file features: 277,752

New columns added:
['file_event_count', 'file_time_span_seconds', 'file_has_creation', 'file_creation_time', 'file_last_bdp_timestamp', 'file_has_bdp', 'file_behavior_counter', 'file_has_tunneling']


In [ ]:
# Cell 8: Calculate Event-Level Features
# Event-level features
usnjrnl_with_features['has_basic_info_changed'] = usnjrnl_with_features['EventInfo'].str.contains('Basic_Info_Changed', na=False).astype(int)
usnjrnl_with_features['has_file_closed'] = usnjrnl_with_features['EventInfo'].str.contains('File_Closed', na=False).astype(int)
usnjrnl_with_features['has_file_created'] = usnjrnl_with_features['EventInfo'].str.contains('File_Created', na=False).astype(int)

# Calculate timestamp differences
usnjrnl_with_features['si_c_diff_seconds'] = abs(
    (usnjrnl_with_features['SI_CreationTime'] - usnjrnl_with_features['EventTime']).dt.total_seconds()
)

usnjrnl_with_features['si_m_diff_seconds'] = abs(
    (usnjrnl_with_features['SI_ModifiedTime'] - usnjrnl_with_features['EventTime']).dt.total_seconds()
)

# Check for zero nanoseconds (timestamps ending in :00)
usnjrnl_with_features['zero_nanoseconds'] = usnjrnl_with_features['TimeStamp(UTC+8)'].str.endswith(':00').astype(int)

print("Event-level features calculated")
print(f"\nSample event features:")
print(usnjrnl_with_features[
    ['File/Directory Name', 'has_basic_info_changed', 'has_file_closed', 
     'si_c_diff_seconds', 'file_behavior_counter']
].head(10))


Event-level features calculated

Sample event features:
  File/Directory Name  has_basic_info_changed  has_file_closed  \
0  3a9f2377dc054e52_0                       0                0   
1  3a9f2377dc054e52_0                       0                0   
2  3a9f2377dc054e52_0                       0                0   
3  3a9f2377dc054e52_0                       0                1   
4  9ff026236b3a5017_0                       0                0   
5  9ff026236b3a5017_0                       0                0   
6  9ff026236b3a5017_0                       0                0   
7  9ff026236b3a5017_0                       0                0   
8  9ff026236b3a5017_0                       0                1   
9  9ff026236b3a5017_1                       0                0   

   si_c_diff_seconds  file_behavior_counter  
0              0.204                      0  
1              0.204                      0  
2              0.204                      0  
3              0.204             

In [ ]:
# Cell 9: Label Events from Ground Truth 
# Extract USN numbers from suspicious.csv
suspicious_usnjrnl = suspicious[suspicious['source'] == 'usnjrnl']

# Extract USN values
suspicious_usns = set(suspicious_usnjrnl['lsn/usn'].astype(int))

print(f"Suspicious USN events in ground truth: {len(suspicious_usns)}")
print(f"Suspicious USNs: {sorted(suspicious_usns)}")

# Label events
usnjrnl_with_features['is_timestomped'] = usnjrnl_with_features['USN'].isin(suspicious_usns).astype(int)

print(f"\nLabeling results:")
print(f"Total events: {len(usnjrnl_with_features):,}")
print(f"Timestomped events: {usnjrnl_with_features['is_timestomped'].sum():,}")
print(f"Benign events: {(usnjrnl_with_features['is_timestomped'] == 0).sum():,}")

# Show labeled timestomped events
if usnjrnl_with_features['is_timestomped'].sum() > 0:
    print(f"\nTimestomped events found:")
    timestomped = usnjrnl_with_features[usnjrnl_with_features['is_timestomped'] == 1]
    print(timestomped[['USN', 'File/Directory Name', 'EventInfo', 'file_behavior_counter']].to_string())


Suspicious USN events in ground truth: 12
Suspicious USNs: [239046272, 239049160, 239049856, 239050536, 239053344, 239054048, 239054704, 239055432, 239057384, 239057592, 239059120, 239059816]

Labeling results:
Total events: 277,752
Timestomped events: 12
Benign events: 277,740

Timestomped events found:
              USN       File/Directory Name                         EventInfo  file_behavior_counter
207295  239046272             DeathToll.jpg  Basic_Info_Changed / File_Closed                      1
207317  239049160              DemLogic.jpg  Basic_Info_Changed / File_Closed                      1
207325  239049856         HoldMyTidePod.jpg  Basic_Info_Changed / File_Closed                      1
207333  239050536             Planning.docx  Basic_Info_Changed / File_Closed                      1
207355  239053344           Huckleberry.png  Basic_Info_Changed / File_Closed                      1
207363  239054048           MyTiredHead.jpg  Basic_Info_Changed / File_Closed           

In [ ]:
# Cell 10: Validate Scoring Logic 
# Check if behavior counter correctly identifies timestomped events
timestomped_events = usnjrnl_with_features[usnjrnl_with_features['is_timestomped'] == 1]

if len(timestomped_events) > 0:
    print("VALIDATION: Behavior Counter for Timestomped Events")
    print("="*60)
    
    for idx, event in timestomped_events.iterrows():
        print(f"\nFile: {event['File/Directory Name']}")
        print(f"USN: {event['USN']}")
        print(f"EventInfo: {event['EventInfo']}")
        print(f"Behavior Counter: {event['file_behavior_counter']}")
        print(f"Has BDP: {event['file_has_bdp']}")
        print(f"Has Tunneling: {event['file_has_tunneling']}")
        print(f"SI-C Diff (seconds): {event['si_c_diff_seconds']:,.0f}")
        
        # Check if counter correctly flagged it
        if event['file_behavior_counter'] >= 2:
            print("STATUS: Correctly flagged as suspicious")
        else:
            print("STATUS: WARNING - Counter too low, check logic")
else:
    print("No timestomped events found in dataset")

# Check false positive rate
benign_high_counter = usnjrnl_with_features[
    (usnjrnl_with_features['is_timestomped'] == 0) &
    (usnjrnl_with_features['file_behavior_counter'] >= 2)
]

print(f"\n\nFalse Positive Check:")
print(f"Benign events with high counter (>= 2): {len(benign_high_counter):,}")
if len(benign_high_counter) > 0:
    print(f"Sample false positives:")
    print(benign_high_counter[['File/Directory Name', 'EventInfo', 'file_behavior_counter']].head())


VALIDATION: Behavior Counter for Timestomped Events

File: DeathToll.jpg
USN: 239046272
EventInfo: Basic_Info_Changed / File_Closed
Behavior Counter: 1
Has BDP: 1
Has Tunneling: 1
SI-C Diff (seconds): 425,079
STATUS: WARNING - Counter too low, check logic

File: DemLogic.jpg
USN: 239049160
EventInfo: Basic_Info_Changed / File_Closed
Behavior Counter: 1
Has BDP: 1
Has Tunneling: 1
SI-C Diff (seconds): 424,887
STATUS: WARNING - Counter too low, check logic

File: HoldMyTidePod.jpg
USN: 239049856
EventInfo: Basic_Info_Changed / File_Closed
Behavior Counter: 1
Has BDP: 1
Has Tunneling: 1
SI-C Diff (seconds): 514,303
STATUS: WARNING - Counter too low, check logic

File: Planning.docx
USN: 239050536
EventInfo: Basic_Info_Changed / File_Closed
Behavior Counter: 1
Has BDP: 1
Has Tunneling: 1
SI-C Diff (seconds): 75,023
STATUS: WARNING - Counter too low, check logic

File: Huckleberry.png
USN: 239053344
EventInfo: Basic_Info_Changed / File_Closed
Behavior Counter: 1
Has BDP: 1
Has Tunneling: 1


In [ ]:
# Cell 11: Save Features 
# Select columns for output
output_columns = [
    # Identifiers
    'USN', 'FileReferenceNumber', 'File/Directory Name', 'FullPath',
    
    # Event info
    'EventInfo', 'EventTime',
    
    # Event-level features
    'has_basic_info_changed', 'has_file_closed', 'has_file_created',
    'si_c_diff_seconds', 'si_m_diff_seconds', 'zero_nanoseconds',
    
    # File-level features
    'file_event_count', 'file_has_creation', 'file_has_bdp',
    'file_behavior_counter', 'file_has_tunneling', 'file_time_span_seconds',
    
    # Label
    'is_timestomped'
]

# Keep only existing columns
output_columns = [col for col in output_columns if col in usnjrnl_with_features.columns]

# Save
usnjrnl_features = usnjrnl_with_features[output_columns].copy()
usnjrnl_features.to_csv(OUTPUT_PATH, index=False)

print(f"Features saved to: {OUTPUT_PATH}")
print(f"Total events: {len(usnjrnl_features):,}")
print(f"Total features: {len(output_columns)}")
print(f"\nFeature list:")
for col in output_columns:
    print(f"  - {col}")


Features saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 2/LoneWolf_UsnJrnl_features.csv
Total events: 277,752
Total features: 19

Feature list:
  - USN
  - FileReferenceNumber
  - File/Directory Name
  - FullPath
  - EventInfo
  - EventTime
  - has_basic_info_changed
  - has_file_closed
  - has_file_created
  - si_c_diff_seconds
  - si_m_diff_seconds
  - zero_nanoseconds
  - file_event_count
  - file_has_creation
  - file_has_bdp
  - file_behavior_counter
  - file_has_tunneling
  - file_time_span_seconds
  - is_timestomped


In [ ]:
# Cell 12: Summary Statistics
print("="*80)
print("USNJRNL FEATURE ENGINEERING SUMMARY - LONEWOLF")
print("="*80)

print(f"\nDataset Statistics:")
print(f"  Original events: {len(usnjrnl):,}")
print(f"  Filtered events: {len(usnjrnl_filtered):,}")
print(f"  Unique files: {usnjrnl_filtered['FileReferenceNumber'].nunique():,}")

print(f"\nFile-Level Statistics:")
print(f"  Files with BDP: {file_features_df['file_has_bdp'].sum():,}")
print(f"  Files with creation events: {file_features_df['file_has_creation'].sum():,}")
print(f"  Files with tunneling: {file_features_df['file_has_tunneling'].sum():,}")

print(f"\nBehavior Counter Distribution:")
print(file_features_df['file_behavior_counter'].value_counts().sort_index())

print(f"\nLabeling Results:")
print(f"  Timestomped events: {usnjrnl_features['is_timestomped'].sum():,}")
print(f"  Benign events: {(usnjrnl_features['is_timestomped'] == 0).sum():,}")
print(f"  Class imbalance: 1:{((usnjrnl_features['is_timestomped'] == 0).sum() / max(usnjrnl_features['is_timestomped'].sum(), 1)):.0f}")

print(f"\nNext Steps:")
print("  1. Validate behavior counter logic")
print("  2. Check if timestomped events are correctly scored")
print("  3. If validation passes, scale to all datasets")
print("  4. Proceed with LogFile feature engineering")

print(f"\n{'='*80}")


USNJRNL FEATURE ENGINEERING SUMMARY - LONEWOLF

Dataset Statistics:
  Original events: 352,849
  Filtered events: 277,752
  Unique files: 66,373

File-Level Statistics:
  Files with BDP: 8,629
  Files with creation events: 54,939
  Files with tunneling: 14,910

Behavior Counter Distribution:
file_behavior_counter
0    26204
1     4016
3    35454
4      699
Name: count, dtype: int64

Labeling Results:
  Timestomped events: 12
  Benign events: 277,740
  Class imbalance: 1:23145

Next Steps:
  1. Validate behavior counter logic
  2. Check if timestomped events are correctly scored
  3. If validation passes, scale to all datasets
  4. Proceed with LogFile feature engineering

